# Проверка гипотезы: дедупликация теряет финальный результат сценария

**Колонки результатов:**
- `DESC_TEXT_1` (в исходном CSV — `DESC_TEXT.1`) — **результат урегулирования по сценарию**
- `DESC_TEXT` — **результат урегулирования ИПУ**

**Гипотеза:** при дедупликации по ключу (ИНН + дата выявления + номер ФП + тип)
`drop_duplicates(keep='first')` оставляет промежуточный результат,
а не финальный. Например, вместо итогового исхода сценария остаётся
«необходим новый план ИПУ».

**Цель:** подтвердить или опровергнуть гипотезу, оценить масштаб проблемы
и найти колонку для сортировки, чтобы гарантированно оставлять последнюю запись.

**Релевантность:** только для отчёта по результатам сценариев. На отчёт по дефолтам не влияет.

## 0. Конфигурация и загрузка

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]

# Колонки-кандидаты для определения порядка записей
ORDER_CANDIDATES = [
    "END_DATE_SCR_FCT",
    "END_DATE_SCR_PLAN",
    "END_EVENT_DATE_FACT",
    "FIRST_END_DATE_EVENT",
    "NEW_PLAN_END_DATE_EVT",
    "DATE_END_FP_SFP",
    "ROW_ID",
]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("OK")

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
raw.columns = raw.columns.str.strip()

# В исходном CSV колонка называется DESC_TEXT.1 — переименовываем
if "DESC_TEXT.1" in raw.columns and "DESC_TEXT_1" not in raw.columns:
    raw = raw.rename(columns={"DESC_TEXT.1": "DESC_TEXT_1"})
    print("Переименована колонка DESC_TEXT.1 → DESC_TEXT_1")

print(f"Загружено: {len(raw):,} строк")

if "VAL" in raw.columns:
    raw = raw[raw["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"После фильтра по источникам: {len(raw):,}")

raw["inn"] = raw["X_INN"].apply(normalize_inn)
raw["dt"] = pd.to_datetime(raw["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
raw = raw[(raw["dt"] >= DATE_FROM) & (raw["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(raw):,}")

raw["segment"] = raw["X_AREA_RESP"].str.strip().map(SEGMENT_MAP)
raw = raw[raw["segment"].notna()].copy()

raw["fp_num"] = raw["NUMBER_FP_SFP"].str.strip()
raw["fp_type"] = raw["TYPE_FP"].str.strip()

# DESC_TEXT_1 = результат сценария, DESC_TEXT = результат ИПУ
raw["scenario"] = raw["DESC_TEXT_1"].fillna("").str.strip()
raw["ipu"]      = raw["DESC_TEXT"].fillna("").str.strip()

print(f"Итого для анализа: {len(raw):,} строк, {raw['inn'].nunique():,} ИНН")

## 1. Масштаб проблемы: сколько карточек ФП имеют дубликаты с разными результатами?

- `DESC_TEXT_1` → `scenario` (результат сценария)
- `DESC_TEXT` → `ipu` (результат ИПУ)

In [ ]:
dup_mask = raw.duplicated(subset=DEDUP_KEY, keep=False)
n_total_cards = raw.drop_duplicates(subset=DEDUP_KEY).shape[0]
n_dup_rows = dup_mask.sum()

dup_groups = raw[dup_mask].groupby(DEDUP_KEY)
n_dup_cards = dup_groups.ngroups

print(f"Всего уникальных карточек ФП: {n_total_cards:,}")
print(f"Карточек с дубликатами (>1 строки): {n_dup_cards:,} ({n_dup_cards/n_total_cards*100:.1f}%)")
print(f"Строк в дубликатных группах: {n_dup_rows:,}")

# Сколько дубликатов в группе
grp_sizes = dup_groups.size()
print(f"\nРазмеры дубликатных групп:")
print(grp_sizes.value_counts().sort_index().to_string())

In [ ]:
# Сколько из них имеют РАЗНЫЕ результаты сценария / ИПУ?
sc_nuniq  = dup_groups["scenario"].nunique()
ipu_nuniq = dup_groups["ipu"].nunique()

diff_sc  = sc_nuniq[sc_nuniq > 1]
diff_ipu = ipu_nuniq[ipu_nuniq > 1]

print(f"Карточек с >1 уникальным DESC_TEXT_1 (сценарий): {len(diff_sc):,} из {n_dup_cards:,}")
print(f"Карточек с >1 уникальным DESC_TEXT   (ИПУ):      {len(diff_ipu):,} из {n_dup_cards:,}")

if n_dup_cards > 0:
    print(f"\n→ {len(diff_sc)/n_dup_cards*100:.1f}% дубликатных карточек имеют расхождение в результате сценария")
    print(f"→ {len(diff_ipu)/n_dup_cards*100:.1f}% дубликатных карточек имеют расхождение в результате ИПУ")

## 2. Сравнение keep='first' vs keep='last'

In [ ]:
first_df = raw.drop_duplicates(subset=DEDUP_KEY, keep="first")
last_df  = raw.drop_duplicates(subset=DEDUP_KEY, keep="last")

# Агрегированная разница по результату СЦЕНАРИЯ (DESC_TEXT_1)
first_vc = first_df["scenario"].value_counts()
last_vc  = last_df["scenario"].value_counts()

all_vals = sorted(set(first_vc.index) | set(last_vc.index))
delta = pd.DataFrame({
    "Результат сценария (DESC_TEXT_1)": all_vals,
    "keep=first": [first_vc.get(v, 0) for v in all_vals],
    "keep=last":  [last_vc.get(v, 0) for v in all_vals],
})
delta["Разница"] = delta["keep=last"] - delta["keep=first"]
delta = delta[delta["Разница"] != 0].sort_values("Разница")

if len(delta) > 0:
    print("Результаты СЦЕНАРИЯ (DESC_TEXT_1): разница между keep=first и keep=last")
    print("(+) = чаще при keep=last, (-) = реже при keep=last\n")
    display(delta.style.hide(axis="index").bar(subset=["Разница"], color=["#d65f5f", "#5fba7d"]))
else:
    print("Разницы между keep=first и keep=last для результата сценария нет.")

In [ ]:
# То же для результата ИПУ (DESC_TEXT)
first_vc_ipu = first_df["ipu"].value_counts()
last_vc_ipu  = last_df["ipu"].value_counts()

all_vals_ipu = sorted(set(first_vc_ipu.index) | set(last_vc_ipu.index))
delta_ipu = pd.DataFrame({
    "Результат ИПУ (DESC_TEXT)": all_vals_ipu,
    "keep=first": [first_vc_ipu.get(v, 0) for v in all_vals_ipu],
    "keep=last":  [last_vc_ipu.get(v, 0) for v in all_vals_ipu],
})
delta_ipu["Разница"] = delta_ipu["keep=last"] - delta_ipu["keep=first"]
delta_ipu = delta_ipu[delta_ipu["Разница"] != 0].sort_values("Разница")

if len(delta_ipu) > 0:
    print("Результаты ИПУ (DESC_TEXT): разница между keep=first и keep=last\n")
    display(delta_ipu.style.hide(axis="index").bar(subset=["Разница"], color=["#d65f5f", "#5fba7d"]))
else:
    print("Разницы между keep=first и keep=last для результата ИПУ нет.")

## 3. Примеры расхождений: first vs last для конкретных карточек

In [ ]:
if len(diff_sc) > 0:
    sample_keys = diff_sc.head(30).index.tolist()

    compare = []
    for key in sample_keys:
        inn_v, fp_v, tp_v, dt_v = key
        grp = raw[
            (raw["inn"] == inn_v) &
            (raw["fp_num"] == fp_v) &
            (raw["fp_type"] == tp_v) &
            (raw["IDENTIFICATION_DATE"] == dt_v)
        ].sort_index()
        if len(grp) >= 2:
            compare.append({
                "ИНН": inn_v[:6] + "...",
                "ФП": fp_v,
                "Тип": tp_v,
                "Строк": len(grp),
                "Сценарий (first)": grp.iloc[0]["scenario"][:70] or "[пусто]",
                "Сценарий (last)":  grp.iloc[-1]["scenario"][:70] or "[пусто]",
                "ИПУ (first)": grp.iloc[0]["ipu"][:70] or "[пусто]",
                "ИПУ (last)":  grp.iloc[-1]["ipu"][:70] or "[пусто]",
            })

    if compare:
        print(f"Примеры расхождений ({len(compare)} карточек):")
        display(pd.DataFrame(compare).style.hide(axis="index"))
else:
    print("Расхождений в результатах сценария нет — гипотеза не подтверждается.")

## 4. Полная история одной карточки ФП (детальный пример)

In [ ]:
# Берём первую карточку с расхождением и показываем ВСЕ её записи
if len(diff_sc) > 0:
    key = diff_sc.index[0]
    inn_v, fp_v, tp_v, dt_v = key
    grp = raw[
        (raw["inn"] == inn_v) &
        (raw["fp_num"] == fp_v) &
        (raw["fp_type"] == tp_v) &
        (raw["IDENTIFICATION_DATE"] == dt_v)
    ].sort_index()

    show_cols = [
        "inn", "fp_num", "fp_type", "IDENTIFICATION_DATE",
        "scenario", "ipu", "SCRIPT",
        "END_DATE_SCR_PLAN", "END_DATE_SCR_FCT",
        "FIRST_END_DATE_EVENT", "END_EVENT_DATE_FACT",
        "NEW_PLAN_END_DATE_EVT", "DATE_END_FP_SFP",
        "VAL_1", "ROW_ID",
    ]
    show_cols = [c for c in show_cols if c in grp.columns]

    print(f"Карточка: ИНН={inn_v}, ФП={fp_v}, тип={tp_v}, дата={dt_v}")
    print(f"Записей: {len(grp)}\n")
    display(grp[show_cols].style.hide(axis="index"))
else:
    print("Нет карточек с расхождениями.")

## 5. Кандидаты для сортировки: заполняемость и разброс внутри дубликатных групп

In [ ]:
dup_data = raw[dup_mask].copy()

results = []
for col in ORDER_CANDIDATES:
    if col not in dup_data.columns:
        results.append((col, "НЕТ В ДАННЫХ", "-", "-", "-"))
        continue

    vals = dup_data[col].fillna("").str.strip()
    filled = (vals != "").sum()
    fill_pct = filled / len(dup_data) * 100

    # Сколько групп имеют разные значения этой колонки?
    nuniq = dup_data.groupby(DEDUP_KEY)[col].nunique()
    n_diff = (nuniq > 1).sum()
    pct_diff = n_diff / n_dup_cards * 100 if n_dup_cards > 0 else 0

    results.append((
        col,
        f"{fill_pct:.1f}%",
        f"{filled:,}",
        f"{n_diff:,}",
        f"{pct_diff:.1f}%",
    ))

cand_df = pd.DataFrame(results, columns=[
    "Колонка", "Заполняемость", "Заполнено строк",
    "Групп с разными значениями", "% от дубликатных групп",
])
print("Кандидаты для определения порядка записей (в дубликатных группах):")
display(cand_df.style.hide(axis="index"))

In [ ]:
# Детальнее: для каждого кандидата-даты — распределение разницы (max - min) внутри группы
date_candidates = [c for c in ORDER_CANDIDATES if c != "ROW_ID" and c in dup_data.columns]

for col in date_candidates:
    dup_data[f"_{col}_dt"] = pd.to_datetime(dup_data[col], dayfirst=True, errors="coerce")

    grp_stats = dup_data.groupby(DEDUP_KEY)[f"_{col}_dt"].agg(["min", "max"])
    grp_stats["diff_days"] = (grp_stats["max"] - grp_stats["min"]).dt.days
    has_diff = grp_stats[grp_stats["diff_days"] > 0]["diff_days"]

    if len(has_diff) > 0:
        print(f"\n{col}: {len(has_diff):,} групп с разными датами")
        print(f"  Разница (дни): mean={has_diff.mean():.0f}, "
              f"median={has_diff.median():.0f}, "
              f"min={has_diff.min():.0f}, max={has_diff.max():.0f}")
    else:
        print(f"\n{col}: нет групп с разными датами (не подходит для сортировки)")

## 6. ROW_ID как порядковый номер

In [ ]:
if "ROW_ID" in dup_data.columns:
    dup_data["_row_id_num"] = pd.to_numeric(dup_data["ROW_ID"], errors="coerce")
    filled_rid = dup_data["_row_id_num"].notna().sum()
    print(f"ROW_ID: {filled_rid:,} числовых из {len(dup_data):,}")

    # Проверяем: внутри дубликатных групп ROW_ID различается?
    rid_nuniq = dup_data.groupby(DEDUP_KEY)["_row_id_num"].nunique()
    n_rid_diff = (rid_nuniq > 1).sum()
    print(f"Групп с разными ROW_ID: {n_rid_diff:,} из {n_dup_cards:,}")

    if n_rid_diff > 0:
        # Проверяем гипотезу: запись с MAX ROW_ID имеет финальный результат?
        # Сравниваем: desc из записи с max(ROW_ID) vs desc из last по порядку CSV
        idx_max_rid = dup_data.groupby(DEDUP_KEY)["_row_id_num"].idxmax()
        max_rid_desc = dup_data.loc[idx_max_rid.dropna()].set_index(DEDUP_KEY)["scenario"]

        last_by_csv = raw.drop_duplicates(subset=DEDUP_KEY, keep="last").set_index(DEDUP_KEY)["scenario"]

        common_idx = max_rid_desc.index.intersection(last_by_csv.index)
        match = (max_rid_desc.loc[common_idx] == last_by_csv.loc[common_idx]).sum()
        print(f"\nСовпадение DESC_TEXT: max(ROW_ID) vs keep=last: {match}/{len(common_idx)}")
        print(f"({match/max(len(common_idx),1)*100:.1f}% совпадений)")
else:
    print("Колонка ROW_ID отсутствует.")

## 7. Итоговая рекомендация

In [ ]:
print("="*70)
print("ИТОГ ПРОВЕРКИ")
print("="*70)

if len(diff_sc) == 0:
    print("\n✓ Гипотеза НЕ подтвердилась.")
    print("  Все дубликаты имеют одинаковый результат сценария.")
    print("  Текущая дедупликация корректна.")
else:
    pct = len(diff_sc) / n_total_cards * 100
    print(f"\n⚠ Гипотеза ПОДТВЕРДИЛАСЬ.")
    print(f"  {len(diff_sc):,} карточек ({pct:.2f}% от общего числа) имеют")
    print(f"  разные результаты сценария (DESC_TEXT_1) в дубликатных записях.")
    print(f"\n  При keep=first и keep=last результаты отличаются")
    print(f"  (см. таблицу в секции 2).")
    print(f"\nРекомендация:")
    print(f"  Выбрать колонку для сортировки из секции 5 (с наибольшим")
    print(f"  % групп с разными значениями) и сортировать по убыванию")
    print(f"  перед дедупликацией.")

## 8. Проверка: DESC_TEXT (ИПУ) заполнен только когда сценарий привёл к ИПУ

ИПУ назначается только при определённых негативных исходах сценария.
Если результат сценария (`DESC_TEXT_1`) **не** является одним из «триггерных»,
то результат ИПУ (`DESC_TEXT`) должен быть пустым.

In [ ]:
# Результаты сценария, которые приводят к назначению ИПУ
IPU_TRIGGER_SCENARIOS = [
    "ФП оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ПУ",
    "ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "Микро_У ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "10_ККБ/МСБ_ФП не устранен в рамках сценария, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ИПУ",
    "ФП не устранен в рамках сценария, оказывает негативное влияние, требует рассмотрения вопроса на УОБ об утверждении Плана устранения ФП",
]

# Работаем с дедуплицированными данными (keep=first — текущее поведение)
df_check = first_df.copy()

df_check["scenario_triggers_ipu"] = df_check["scenario"].isin(IPU_TRIGGER_SCENARIOS)
df_check["ipu_filled"] = df_check["ipu"] != ""

n_total = len(df_check)
n_trigger = df_check["scenario_triggers_ipu"].sum()
n_ipu_filled = df_check["ipu_filled"].sum()

print(f"Всего карточек ФП (после дедупликации keep=first): {n_total:,}")
print(f"Сценарий → ИПУ (триггерный исход): {n_trigger:,} ({n_trigger/n_total*100:.1f}%)")
print(f"DESC_TEXT (ИПУ) заполнен: {n_ipu_filled:,} ({n_ipu_filled/n_total*100:.1f}%)")

# Кросс-таблица
cross = pd.crosstab(
    df_check["scenario_triggers_ipu"].map({True: "Сценарий → ИПУ", False: "Сценарий без ИПУ"}),
    df_check["ipu_filled"].map({True: "ИПУ заполнен", False: "ИПУ пуст"}),
    margins=True, margins_name="Итого"
)
print("\nКросс-таблица:")
display(cross)

# Аномалии: ИПУ заполнен, но сценарий НЕ триггерный
anomalies = df_check[~df_check["scenario_triggers_ipu"] & df_check["ipu_filled"]]
print(f"\n⚠ Аномалии: ИПУ заполнен при НЕтриггерном сценарии: {len(anomalies):,}")

if len(anomalies) > 0:
    print(f"\nРезультаты сценария у аномалий:")
    anom_sc = anomalies["scenario"].value_counts()
    for val, cnt in anom_sc.items():
        print(f"  {cnt:>5,}  {val or '[пусто]'}")

    print(f"\nРезультаты ИПУ у аномалий:")
    anom_ipu = anomalies["ipu"].value_counts()
    for val, cnt in anom_ipu.head(20).items():
        print(f"  {cnt:>5,}  {val}")

In [ ]:
# То же самое, но для keep=last — уменьшается ли количество аномалий?
df_check_last = last_df.copy()
df_check_last["scenario_triggers_ipu"] = df_check_last["scenario"].isin(IPU_TRIGGER_SCENARIOS)
df_check_last["ipu_filled"] = df_check_last["ipu"] != ""

anomalies_last = df_check_last[~df_check_last["scenario_triggers_ipu"] & df_check_last["ipu_filled"]]

print(f"Аномалии (ИПУ заполнен при НЕтриггерном сценарии):")
print(f"  keep=first: {len(anomalies):,}")
print(f"  keep=last:  {len(anomalies_last):,}")
diff_anom = len(anomalies) - len(anomalies_last)
if diff_anom > 0:
    print(f"\n  → keep=last уменьшает аномалии на {diff_anom:,} записей")
elif diff_anom < 0:
    print(f"\n  → keep=last увеличивает аномалии на {-diff_anom:,} записей")
else:
    print(f"\n  → Разницы нет")

# Кросс-таблица для keep=last
cross_last = pd.crosstab(
    df_check_last["scenario_triggers_ipu"].map({True: "Сценарий → ИПУ", False: "Сценарий без ИПУ"}),
    df_check_last["ipu_filled"].map({True: "ИПУ заполнен", False: "ИПУ пуст"}),
    margins=True, margins_name="Итого"
)
print("\nКросс-таблица (keep=last):")
display(cross_last)

## 9. Заполняемость ИПУ-колонок при триггерных сценариях

Колонки ИПУ (`FIRST_END_DATE_EVENT`, `END_EVENT_DATE_FACT`, `NEW_PLAN_END_DATE_EVT`, `DESC_TEXT`)
должны заполняться только когда сценарий завершился негативно и назначается ИПУ.
При триггерных сценариях ожидаем заполняемость ~100%.

In [ ]:
IPU_DATE_COLS = ["FIRST_END_DATE_EVENT", "END_EVENT_DATE_FACT", "NEW_PLAN_END_DATE_EVT"]
IPU_RESULT_COL = "ipu"

# Работаем на СЫРЫХ данных (до дедупликации), чтобы видеть полную картину
raw["_is_trigger"] = raw["scenario"].isin(IPU_TRIGGER_SCENARIOS)

trigger_rows    = raw[raw["_is_trigger"]]
nontrigger_rows = raw[~raw["_is_trigger"]]

print(f"Записей с триггерным сценарием: {len(trigger_rows):,}")
print(f"Записей без триггерного сценария: {len(nontrigger_rows):,}\n")

# Заполняемость ИПУ-колонок: триггерные vs нетриггерные
fill_rows = []
for col in IPU_DATE_COLS + [IPU_RESULT_COL]:
    if col == IPU_RESULT_COL:
        t_filled = (trigger_rows[col] != "").sum()
        nt_filled = (nontrigger_rows[col] != "").sum()
    else:
        t_filled = trigger_rows[col].fillna("").str.strip().ne("").sum()
        nt_filled = nontrigger_rows[col].fillna("").str.strip().ne("").sum()

    t_pct = t_filled / max(len(trigger_rows), 1) * 100
    nt_pct = nt_filled / max(len(nontrigger_rows), 1) * 100

    fill_rows.append({
        "Колонка": col if col != IPU_RESULT_COL else "DESC_TEXT (результат ИПУ)",
        "Триггерные: заполнено": f"{t_filled:,}",
        "Триггерные: %": f"{t_pct:.1f}%",
        "Нетриггерные: заполнено": f"{nt_filled:,}",
        "Нетриггерные: %": f"{nt_pct:.1f}%",
    })

print("Заполняемость ИПУ-колонок: триггерные vs нетриггерные сценарии")
display(pd.DataFrame(fill_rows).style.hide(axis="index"))

In [ ]:
# Детально: заполняемость по каждому из 5 триггерных сценариев
detail_rows = []
for sc_text in IPU_TRIGGER_SCENARIOS:
    subset = raw[raw["scenario"] == sc_text]
    n = len(subset)
    if n == 0:
        continue
    row = {"Сценарий": sc_text[:80], "Записей": f"{n:,}"}
    for col in IPU_DATE_COLS + [IPU_RESULT_COL]:
        if col == IPU_RESULT_COL:
            filled = (subset[col] != "").sum()
        else:
            filled = subset[col].fillna("").str.strip().ne("").sum()
        row[col if col != IPU_RESULT_COL else "DESC_TEXT (ИПУ)"] = f"{filled/n*100:.0f}%"
    detail_rows.append(row)

print("Заполняемость ИПУ-колонок по каждому триггерному сценарию:")
display(pd.DataFrame(detail_rows).style.hide(axis="index"))

In [ ]:
# Обратная проверка: если ИПУ-колонка заполнена — какой % имеет триггерный сценарий?
print("Обратная проверка: среди записей с заполненными ИПУ-колонками — какой % триггерных?\n")

for col in IPU_DATE_COLS + [IPU_RESULT_COL]:
    label = col if col != IPU_RESULT_COL else "DESC_TEXT (ИПУ)"
    if col == IPU_RESULT_COL:
        mask = raw[col] != ""
    else:
        mask = raw[col].fillna("").str.strip().ne("")

    filled_total = mask.sum()
    if filled_total == 0:
        print(f"  {label}: 0 заполненных строк")
        continue

    filled_trigger = (mask & raw["_is_trigger"]).sum()
    pct = filled_trigger / filled_total * 100
    print(f"  {label}: {filled_total:,} заполненных, "
          f"из них {filled_trigger:,} ({pct:.1f}%) с триггерным сценарием")

## 10. Нейтральные результаты — признак промежуточной записи

Гипотеза: карточки с нейтральным результатом сценария при `keep='first'`
на самом деле имеют другие (более поздние) записи с финальным результатом.
Нейтральные результаты — промежуточные состояния, а не итоговый исход.

Используем 27 нейтральных значений из `scenario_results_classification.txt`.

In [ ]:
NEUTRAL_SCENARIOS = [
    "ФП не устранен, не оказывает негативного влияния, требует рассмотрения вопроса на УОБ с целью снятия ФП с контроля",
    "Активный (срок окончания, предусмотренный Порядком не наступил, находится в стадии реализации)",
    "11_ККБ/МСБ_ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "4_МСБ_Право Банка реализовано, ФП снят с контроля",
    "5_МСБ_Право Банка не реализовано, ФП снят с контроля",
    "6_ККБ/МСБ_Техническое закрытие ФП",
    "7_ККБ/МСБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "Активный (срок окончания, предусмотренный Порядком наступил, результат реализации по состоянию на дату Реестра отсутствует)",
    "Микро_У ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "Микро_У Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "8_ККБ/МСБ_В отношении ФП принято решение УОБ",
    "ДКБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "5_ККБ/МСБ_ФП не требует устранения, право Банка не реализовано",
    "ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "4_ККБ/МСБ_ФП не требует устранения, право Банка реализовано",
    "Условие не выполнено, в Н2.0 зафиксировано нарушение Условия (ответственное подразделение за закрытие КП)",
    "ФП не требует устранения, право Банка не реализовано",
    "Установлена коммерческая ставка",
    "Микро_У Фактор устранен - решение УОБ о снятии фактора с контроля",
    "9_ККБ/МСБ_В отношении ФП принято решение УЛ",
    "ДКБ_Информация проверена, выявлен соответствующий ФП/СФП",
    "Микро_У Принятие решения УЛ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ/УЛ о снятии ФП с контроля",
    "По ФП реализовано право Банка",
]

POSITIVE_SCENARIOS = [
    "ФП/СФП устранен", "ФП/СФП урегулирован", "Микро_У ФП урегулирован",
    "МСБ_Принято решение УОБ о снятии ФП с контроля",
    "МСБ_Принято решение УЛ о снятии ФП с контроля",
    "1_ККБ/МСБ_ФП устранен", "СФП устранен",
    "Микро_У Требуется принятие решения УЛ о снятии фактора с контроля",
    "Принято решение УЛ о снятии ФП с контроля",
    "Микро_У Фактор устранен", "Денежная санкция уплачена",
    "ФП/СФП устранен (договор закрыт)", "Условие выполнено",
    "2_ККБ/МСБ_СФП устранен",
    "ДКБ_Принято решение УЛ о снятии ФП с контроля",
    "ДКБ_Принято решение УОБ об урегулировании ФП",
    "ДКБ_Принято решение УОБ о снятии ФП с контроля",
    "ФП устранен (договор закрыт)",
    "ДКБ_Не требуется решение УЛ/УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УОБ о снятии фактора с контроля",
    "Микро_У ФП устранен", "ФП устранен",
    "ДКБ_ФП не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, возможно снятие ФП с контроля",
    "Принято решение УОБ о снятии ФП с контроля",
    "Микро_У СФП устранен",
    "ФП урегулирован, требуется принятие решения УОБ о снятии ФП с контроля",
    "ДКБ_СФП устранен",
    "Микро_У Принято решение УОБ о снятии ФП с контроля",
    "Микро_У СФП урегулирован",
    "Микро_У Принято решение УЛ о снятии ФП с контроля",
]

ERROR_SCENARIOS = [
    "0_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "ДКБ_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "Микро_У Информация проверена, отсутствуют основания для выявления ФП/СФП",
]

# Все негативные = IPU_TRIGGER_SCENARIOS + остальные негативные из классификации
NEGATIVE_SCENARIOS = [
    "ФП/СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "3_ККБ/МСБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ПУ",
    "Микро_У Фактор не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "Микро_У ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "ДКБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "12_ККБ/МСБ_ФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "Микро_У СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "10_ККБ/МСБ_ФП не устранен в рамках сценария, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ИПУ",
    "ДКБ_ФП оказывает негативное влияние, требует рассмотрения УОБ вопроса о признании задолженности проблемной",
    "Микро_У Фактор не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП не устранен в рамках сценария, оказывает негативное влияние, требует рассмотрения вопроса на УОБ об утверждении Плана устранения ФП",
    'ФП не устранен. Лимит снижен до "0". Лимит восстановлению не подлежит.',
    "Микро_У Техническое закрытие ФП в связи с запуском нового сценария по СФП (СФП 15.2У/15.2.1У)",
]

print(f"Нейтральных вариантов: {len(NEUTRAL_SCENARIOS)}")
print(f"Положительных: {len(POSITIVE_SCENARIOS)}, Отрицательных: {len(NEGATIVE_SCENARIOS)}, Ошибок: {len(ERROR_SCENARIOS)}")
print(f"Всего: {len(NEUTRAL_SCENARIOS)+len(POSITIVE_SCENARIOS)+len(NEGATIVE_SCENARIOS)+len(ERROR_SCENARIOS)}")

In [ ]:
# Карточки с нейтральным результатом при keep='first'
neutral_first = first_df[first_df["scenario"].isin(NEUTRAL_SCENARIOS)].copy()
n_neutral_first = len(neutral_first)
print(f"Карточек с нейтральным сценарием при keep=first: {n_neutral_first:,}\n")

# Из них — сколько имеют дубликаты (т.е. другие записи в raw)?
grp_counts = raw.groupby(DEDUP_KEY).size().reset_index(name="_grp_size")
neutral_merged = neutral_first.merge(grp_counts, on=DEDUP_KEY, how="left")

has_dups = (neutral_merged["_grp_size"] > 1).sum()
no_dups  = n_neutral_first - has_dups

print(f"Из них имеют >1 записи в сырых данных: {has_dups:,} ({has_dups/max(n_neutral_first,1)*100:.1f}%)")
print(f"Одна запись (дубликатов нет): {no_dups:,}")

In [ ]:
# У карточек с нейтральным first — какой результат при keep='last'?
neutral_keys_df = neutral_first[DEDUP_KEY].copy()
neutral_last = neutral_keys_df.merge(last_df[DEDUP_KEY + ["scenario"]], on=DEDUP_KEY, how="inner",
                                      suffixes=("", "_last"))
neutral_last.rename(columns={"scenario": "scenario_last"}, inplace=True)

_neut_set = set(NEUTRAL_SCENARIOS)
_pos_set  = set(POSITIVE_SCENARIOS)
_neg_set  = set(NEGATIVE_SCENARIOS)
_err_set  = set(ERROR_SCENARIOS)

def classify(val):
    if val in _neut_set:
        return "нейтральный"
    elif val in _pos_set:
        return "положительный"
    elif val in _neg_set:
        return "отрицательный"
    elif val in _err_set:
        return "ошибка"
    elif val == "":
        return "[пусто]"
    return "неклассифицированный"

neutral_last["class_last"] = neutral_last["scenario_last"].apply(classify)

false_neutral = neutral_last[neutral_last["class_last"] != "нейтральный"]
n_false = len(false_neutral)
pct_false = n_false / max(n_neutral_first, 1) * 100

print(f"Карточек с нейтральным сценарием (keep=first): {n_neutral_first:,}")
print(f"Из них при keep=last — НЕнейтральный результат: {n_false:,} ({pct_false:.1f}%)")
print(f"  → Эти карточки при keep=first теряют финальный результат\n")

print("Класс финального результата (last) у «ложных нейтральных»:")
class_vc = false_neutral["class_last"].value_counts()
for cls, cnt in class_vc.items():
    print(f"  {cls:>20s}: {cnt:,}")
print()

In [ ]:
# Таблица переходов: нейтральный (first) → конкретный результат (last)
# Показываем только «ложные нейтральные» — где last отличается от нейтрального
if n_false > 0:
    trans = false_neutral.copy()

    # Берём конкретные значения first и last
    trans_first = neutral_first.set_index(DEDUP_KEY)["scenario"]
    trans = trans.set_index(DEDUP_KEY)
    trans["scenario_first"] = trans_first.reindex(trans.index).values
    trans = trans.reset_index()

    # Кросс-таблица: first (конкретное значение) → class_last (класс)
    cross_trans = pd.crosstab(
        trans["scenario_first"].str[:60],
        trans["class_last"],
        margins=True, margins_name="Итого"
    )
    print("Переходы: нейтральный результат (first) → класс финального результата (last)")
    display(cross_trans)
else:
    print("«Ложных нейтральных» не обнаружено.")

In [ ]:
# Детально: top конкретных переходов first → last
if n_false > 0:
    trans_detail = false_neutral.copy()
    trans_detail = trans_detail.set_index(DEDUP_KEY)
    trans_detail["scenario_first"] = trans_first.reindex(trans_detail.index).values
    trans_detail = trans_detail.reset_index()

    pairs = trans_detail.groupby(["scenario_first", "scenario_last"]).size().reset_index(name="Кол-во")
    pairs = pairs.sort_values("Кол-во", ascending=False)
    pairs["first (сокр.)"] = pairs["scenario_first"].str[:60]
    pairs["last (сокр.)"]  = pairs["scenario_last"].str[:60]

    print(f"Топ-30 конкретных переходов нейтральный → финальный:")
    display(
        pairs[["first (сокр.)", "last (сокр.)", "Кол-во"]].head(30)
        .style.hide(axis="index")
        .bar(subset=["Кол-во"], color="#5fba7d")
    )
else:
    print("Переходов нет.")

In [ ]:
# Итог секции 10
print("=" * 70)
print("ИТОГ: нейтральные результаты как промежуточные записи")
print("=" * 70)

if n_false > 0:
    print(f"\n⚠ Из {n_neutral_first:,} карточек с нейтральным сценарием (keep=first),")
    print(f"  {n_false:,} ({pct_false:.1f}%) при keep=last имеют ДРУГОЙ (ненейтральный) класс.")
    print(f"\n  Это подтверждает, что нейтральные результаты часто являются")
    print(f"  промежуточными состояниями, а keep=first теряет финальный исход.")

    n_to_pos = (false_neutral["class_last"] == "положительный").sum()
    n_to_neg = (false_neutral["class_last"] == "отрицательный").sum()
    n_to_other = n_false - n_to_pos - n_to_neg
    print(f"\n  Из «ложных нейтральных» переходят:")
    print(f"    → положительный: {n_to_pos:,}")
    print(f"    → отрицательный: {n_to_neg:,}")
    print(f"    → другое:        {n_to_other:,}")
else:
    print("\n✓ Нейтральные результаты при keep=first и keep=last совпадают.")
    print("  Проблемы с промежуточными записями не выявлено.")

## 11. Гипотеза: выбор записи по самой поздней дате сценария/ИПУ

Маркер последней записи — **максимальная** дата из:
- `END_DATE_SCR_FCT` или `END_DATE_SCR_PLAN` (сценарий)
- `END_EVENT_DATE_FACT` или `FIRST_END_DATE_EVENT` или `NEW_PLAN_END_DATE_EVT` (ИПУ)

Для каждой дубликатной группы выбираем запись с самой поздней из этих дат
и сравниваем результат с `keep='first'`: какой процент карточек сменит класс
результата сценария (нейтральный → положительный/отрицательный и т.д.)?

In [ ]:
LATEST_DATE_COLS = [
    "END_DATE_SCR_FCT", "END_DATE_SCR_PLAN",
    "END_EVENT_DATE_FACT", "FIRST_END_DATE_EVENT", "NEW_PLAN_END_DATE_EVT",
]

raw_dt = raw.copy()
for col in LATEST_DATE_COLS:
    raw_dt[f"_{col}_dt"] = pd.to_datetime(raw_dt[col], dayfirst=True, errors="coerce")

dt_cols = [f"_{c}_dt" for c in LATEST_DATE_COLS]
raw_dt["_max_date"] = raw_dt[dt_cols].max(axis=1)

filled_max = raw_dt["_max_date"].notna().sum()
print(f"Записей с хотя бы одной датой из {len(LATEST_DATE_COLS)} колонок: "
      f"{filled_max:,} из {len(raw_dt):,} ({filled_max/len(raw_dt)*100:.1f}%)")

# Для каждой группы берём запись с max(_max_date); при NaT — fallback на первую запись
raw_dt["_has_date"] = raw_dt["_max_date"].notna()
raw_dt_sorted = raw_dt.sort_values(["_has_date", "_max_date"], ascending=[False, False])
latest_df = raw_dt_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()

n_with_date = latest_df["_has_date"].sum()
n_fallback  = len(latest_df) - n_with_date

print(f"Карточек по max(дата): {len(latest_df):,}")
print(f"  из них с датой: {n_with_date:,}")
print(f"  из них fallback (нет ни одной даты, взята первая): {n_fallback:,}")

In [ ]:
# Сравниваем результат сценария: keep='first' vs max(дата)
comp = first_df[DEDUP_KEY + ["scenario"]].merge(
    latest_df[DEDUP_KEY + ["scenario"]],
    on=DEDUP_KEY, suffixes=("_first", "_latest")
)

comp["class_first"]  = comp["scenario_first"].apply(classify)
comp["class_latest"] = comp["scenario_latest"].apply(classify)

changed = comp[comp["class_first"] != comp["class_latest"]]
n_changed = len(changed)
pct_changed = n_changed / max(len(comp), 1) * 100

print(f"Всего сравнено карточек: {len(comp):,}")
print(f"Сменился класс результата: {n_changed:,} ({pct_changed:.1f}%)\n")

if n_changed > 0:
    trans = pd.crosstab(changed["class_first"], changed["class_latest"],
                        margins=True, margins_name="Итого")
    print("Матрица переходов (строки = first, столбцы = max(дата)):")
    display(trans)

In [ ]:
# Агрегированная разница по конкретным значениям результата сценария
first_vc_11 = comp["scenario_first"].value_counts()
latest_vc_11 = comp["scenario_latest"].value_counts()

all_vals_11 = sorted(set(first_vc_11.index) | set(latest_vc_11.index))
delta_11 = pd.DataFrame({
    "Результат сценария": all_vals_11,
    "keep=first": [first_vc_11.get(v, 0) for v in all_vals_11],
    "max(дата)":  [latest_vc_11.get(v, 0) for v in all_vals_11],
})
delta_11["Разница"] = delta_11["max(дата)"] - delta_11["keep=first"]
delta_11 = delta_11[delta_11["Разница"] != 0].sort_values("Разница")

if len(delta_11) > 0:
    print("Результаты сценария: keep=first vs max(дата)")
    print("(+) = чаще при max(дата), (-) = реже\n")
    display(delta_11.style.hide(axis="index").bar(subset=["Разница"], color=["#d65f5f", "#5fba7d"]))
else:
    print("Разницы между keep=first и max(дата) нет.")

## 12. Гипотеза: выбор записи по ближайшей дате к DATE_END_FP_SFP

`DATE_END_FP_SFP` — дата снятия ФП/СФП с контроля — одинакова для всех дубликатов
одной карточки, но разные записи имеют разные даты сценария/ИПУ.

Выбираем запись, у которой **любая** из дат
(`END_DATE_SCR_FCT`, `END_DATE_SCR_PLAN`, `END_EVENT_DATE_FACT`,
`FIRST_END_DATE_EVENT`, `NEW_PLAN_END_DATE_EVT`)
ближе всего к `DATE_END_FP_SFP`. Логика: запись, ближайшая к дате закрытия —
наиболее финальная.

In [ ]:
raw_dt["_date_end_fp"] = pd.to_datetime(raw_dt["DATE_END_FP_SFP"], dayfirst=True, errors="coerce")

# Для каждой записи вычисляем минимальное абсолютное расстояние
# от любой из 5 дат до DATE_END_FP_SFP
min_dist_cols = []
for col in LATEST_DATE_COLS:
    dcol = f"_{col}_dt"
    dist_col = f"_dist_{col}"
    raw_dt[dist_col] = (raw_dt[dcol] - raw_dt["_date_end_fp"]).abs()
    min_dist_cols.append(dist_col)

raw_dt["_min_dist"] = raw_dt[min_dist_cols].min(axis=1)

has_dist = raw_dt["_min_dist"].notna().sum()
has_end_fp = raw_dt["_date_end_fp"].notna().sum()
print(f"Записей с DATE_END_FP_SFP: {has_end_fp:,} из {len(raw_dt):,}")
print(f"Записей с вычисленным расстоянием: {has_dist:,}")

# Для каждой группы выбираем запись с минимальным расстоянием
raw_dt["_has_dist"] = raw_dt["_min_dist"].notna()
raw_dt_sorted2 = raw_dt.sort_values(["_has_dist", "_min_dist"], ascending=[False, True])
closest_df = raw_dt_sorted2.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()

n_with_dist = closest_df["_has_dist"].sum()
n_fallback2 = len(closest_df) - n_with_dist

print(f"\nКарточек по min(расстояние до DATE_END_FP_SFP): {len(closest_df):,}")
print(f"  с расстоянием: {n_with_dist:,}")
print(f"  fallback (нет даты/DATE_END_FP_SFP): {n_fallback2:,}")

In [ ]:
# Сравниваем результат: keep='first' vs closest-to-DATE_END_FP_SFP
comp2 = first_df[DEDUP_KEY + ["scenario"]].merge(
    closest_df[DEDUP_KEY + ["scenario"]],
    on=DEDUP_KEY, suffixes=("_first", "_closest")
)

comp2["class_first"]   = comp2["scenario_first"].apply(classify)
comp2["class_closest"] = comp2["scenario_closest"].apply(classify)

changed2 = comp2[comp2["class_first"] != comp2["class_closest"]]
n_changed2 = len(changed2)
pct_changed2 = n_changed2 / max(len(comp2), 1) * 100

print(f"Всего сравнено карточек: {len(comp2):,}")
print(f"Сменился класс результата: {n_changed2:,} ({pct_changed2:.1f}%)\n")

if n_changed2 > 0:
    trans2 = pd.crosstab(changed2["class_first"], changed2["class_closest"],
                         margins=True, margins_name="Итого")
    print("Матрица переходов (строки = first, столбцы = ближайшая к DATE_END_FP_SFP):")
    display(trans2)

In [ ]:
# Агрегированная разница по конкретным значениям
first_vc_12 = comp2["scenario_first"].value_counts()
closest_vc_12 = comp2["scenario_closest"].value_counts()

all_vals_12 = sorted(set(first_vc_12.index) | set(closest_vc_12.index))
delta_12 = pd.DataFrame({
    "Результат сценария": all_vals_12,
    "keep=first": [first_vc_12.get(v, 0) for v in all_vals_12],
    "closest(DATE_END)": [closest_vc_12.get(v, 0) for v in all_vals_12],
})
delta_12["Разница"] = delta_12["closest(DATE_END)"] - delta_12["keep=first"]
delta_12 = delta_12[delta_12["Разница"] != 0].sort_values("Разница")

if len(delta_12) > 0:
    print("Результаты сценария: keep=first vs closest(DATE_END_FP_SFP)")
    print("(+) = чаще при closest, (-) = реже\n")
    display(delta_12.style.hide(axis="index").bar(subset=["Разница"], color=["#d65f5f", "#5fba7d"]))
else:
    print("Разницы нет.")

In [ ]:
# Сводное сравнение 4 методов дедупликации
methods = {
    "keep=first (текущий)": first_df,
    "keep=last": last_df,
    "max(дата сценария/ИПУ)": latest_df,
    "closest(DATE_END_FP_SFP)": closest_df,
}

summary_rows = []
for name, df_m in methods.items():
    sc_vc = df_m["scenario"].apply(classify).value_counts()
    row = {"Метод": name}
    for cls in ["положительный", "отрицательный", "нейтральный", "ошибка", "[пусто]", "неклассифицированный"]:
        row[cls] = sc_vc.get(cls, 0)
    row["Всего"] = len(df_m)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
print("Распределение классов результата сценария по методу дедупликации:")
display(summary.style.hide(axis="index"))

print("\nВ процентах:")
pct = summary.copy()
for cls in ["положительный", "отрицательный", "нейтральный", "ошибка", "[пусто]", "неклассифицированный"]:
    pct[cls] = (pct[cls] / pct["Всего"] * 100).round(1).astype(str) + "%"
display(pct.drop(columns=["Всего"]).style.hide(axis="index"))

## 13. Фильтрация по финальному результату вместо дедупликации

### 13.1. Только положительные и отрицательные результаты

Вместо выбора «первой» или «последней» записи — оставляем только записи,
у которых результат сценария является **положительным** или **отрицательным**
(т.е. финальным). Сколько уникальных ФП мы при этом теряем?

### 13.2. Положительные + отрицательные + отдельные нейтральные

Некоторые «нейтральные» результаты по сути тоже являются финальными
(право реализовано, ФП снят с контроля и т.д.).
Добавляем к фильтру 8 таких результатов и проверяем потери.

In [ ]:
# 13.1 — Только положительные и отрицательные
_pos_neg = set(POSITIVE_SCENARIOS) | set(NEGATIVE_SCENARIOS)

raw["_class"] = raw["scenario"].apply(classify)

# Все уникальные карточки (базовая линия)
all_cards = raw.drop_duplicates(subset=DEDUP_KEY)
n_all = len(all_cards)

# Записи с положительным или отрицательным результатом
pos_neg_rows = raw[raw["scenario"].isin(_pos_neg)]
pos_neg_cards = pos_neg_rows.drop_duplicates(subset=DEDUP_KEY)
n_pos_neg = len(pos_neg_cards)

n_lost_1 = n_all - n_pos_neg
pct_lost_1 = n_lost_1 / n_all * 100

print("13.1. Фильтр: только положительные + отрицательные результаты")
print(f"  Всего уникальных карточек ФП: {n_all:,}")
print(f"  Карточек с полож./отриц. результатом: {n_pos_neg:,}")
print(f"  Потеряно: {n_lost_1:,} ({pct_lost_1:.1f}%)\n")

# Какие классы у потерянных?
lost_df_1 = all_cards.merge(pos_neg_cards[DEDUP_KEY].drop_duplicates(),
                            on=DEDUP_KEY, how="left", indicator=True) \
                     .query("_merge == 'left_only'").drop(columns="_merge")

lost_classes_1 = lost_df_1["_class"].value_counts()
print("Классы потерянных карточек (keep=first):")
for cls, cnt in lost_classes_1.items():
    print(f"  {cls:>25s}: {cnt:,}")

# Топ конкретных результатов у потерянных
print(f"\nТоп-15 конкретных результатов потерянных карточек:")
for val, cnt in lost_df_1["scenario"].value_counts().head(15).items():
    print(f"  {cnt:>5,}  {val or '[пусто]'}")

In [ ]:
# 13.2 — Положительные + отрицательные + 8 «финальных» нейтральных
FINAL_NEUTRAL = [
    "4_МСБ_Право Банка реализовано, ФП снят с контроля",
    "5_МСБ_Право Банка не реализовано, ФП снят с контроля",
    "5_ККБ/МСБ_ФП не требует устранения, право Банка не реализовано",
    "ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "4_ККБ/МСБ_ФП не требует устранения, право Банка реализовано",
    "ФП не требует устранения, право Банка не реализовано",
    "Установлена коммерческая ставка",
    "По ФП реализовано право Банка",
]

_accepted_2 = _pos_neg | set(FINAL_NEUTRAL)

accepted_rows_2 = raw[raw["scenario"].isin(_accepted_2)]
accepted_cards_2 = accepted_rows_2.drop_duplicates(subset=DEDUP_KEY)
n_accepted_2 = len(accepted_cards_2)

n_lost_2 = n_all - n_accepted_2
pct_lost_2 = n_lost_2 / n_all * 100

print("13.2. Фильтр: положительные + отрицательные + 8 финальных нейтральных")
print(f"  Всего уникальных карточек ФП: {n_all:,}")
print(f"  Карточек с допустимым результатом: {n_accepted_2:,}")
print(f"  Потеряно: {n_lost_2:,} ({pct_lost_2:.1f}%)\n")

# Какие классы у потерянных?
lost_df_2 = all_cards.merge(accepted_cards_2[DEDUP_KEY].drop_duplicates(),
                            on=DEDUP_KEY, how="left", indicator=True) \
                     .query("_merge == 'left_only'").drop(columns="_merge")

lost_classes_2 = lost_df_2["_class"].value_counts()
print("Классы потерянных карточек (keep=first):")
for cls, cnt in lost_classes_2.items():
    print(f"  {cls:>25s}: {cnt:,}")

print(f"\nТоп-15 конкретных результатов потерянных карточек:")
for val, cnt in lost_df_2["scenario"].value_counts().head(15).items():
    print(f"  {cnt:>5,}  {val or '[пусто]'}")

In [ ]:
# Сводка: потери при разных вариантах фильтрации
print("=" * 70)
print("СВОДКА: потери уникальных карточек ФП при фильтрации по результату")
print("=" * 70)
print(f"\n  Базовая линия (все уникальные карточки): {n_all:,}")
print(f"\n  13.1  Только полож. + отриц.:")
print(f"        Остаётся: {n_pos_neg:,} | Потеряно: {n_lost_1:,} ({pct_lost_1:.1f}%)")
print(f"\n  13.2  Полож. + отриц. + 8 финальных нейтральных:")
print(f"        Остаётся: {n_accepted_2:,} | Потеряно: {n_lost_2:,} ({pct_lost_2:.1f}%)")
print(f"\n  Разница между 13.1 и 13.2: {n_lost_1 - n_lost_2:,} карточек спасены"
      f" добавлением 8 нейтральных результатов")